# Ratings.CSV 

Este notebook documenta el análisis exploratorio y limpieza de **ratings.csv** de MovieLens.


**Entrada**: ratings.csv \
**Objetivos**: lectura, validación, limpieza y transformación \
**Salida**: ratings_clean.parquet


## Descripción del proceso

**Análisis y comprensión**
- userId: identificador único.
- movieId: identificador entero.
- rating: decimal.
- timestamp: fechas entre "1996-03-26" y "2018-09-26". Formato actual es en segundos.

**Validación**
- La combinación de userId y movieId deben ser únicos
- Fechas entre "1996-03-26" y "2018-09-26" 
- Rating: incluido en [0.5,5] saltos de 0.5

**Limpieza**
- Registros duplicados de calificaciones mismo usuario y película, se guarda la más reciente.
- No se permiten valores nulos, se procede a eliminar el registro. 
- Ratings fuera de rango válido se elimina el registro. 


**Transformación**
- Conversión de fechas a datetime
- Salida archivo ratings.parquet


In [12]:
import pandas as pd
import numpy as np

## Análisis y comprensión del dataset 

In [13]:
# Lectura de datos
ratings = pd.read_csv("../data/01_raw/movielens/ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [14]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


## Validación

Registros duplicados

In [15]:
ratings[ratings[['userId', 'movieId']].duplicated()]
# no se observan

,userId,movieId,rating,timestamp


Validación de fechas.   
La columna timestamp viene en formato timestamp Unix en segundos que han pasado desde 1 enero de 1970.

In [16]:
fechas = pd.to_datetime(ratings['timestamp'], unit="s")  
fechas.between("1996-03-26", "2018-09-26", inclusive="both")
fechas[~fechas.between("1996-03-26", "2018-09-26", inclusive="both")]
#todas los registros de fechas dentro del en rango contemplado

Series([], Name: timestamp, dtype: datetime64[s])

Los ratings deben de estar dentro del intervalo [0.5,5] con saltos de 0.5

In [17]:
escalaRating = np.arange(0.5,5.5,0.5)
ratings[~ratings['rating'].isin(escalaRating)]
# todos los registros cumplen la regla

,userId,movieId,rating,timestamp


# Limpieza


Si hubiese duplicados, se conserva el registro más reciente.

In [18]:
ratings.sort_values('timestamp').drop_duplicates(subset=['movieId', 'userId'], keep='last')

,userId,movieId,rating,timestamp
66669,429,165,4.0,828124615
66719,429,595,5.0,828124615
66713,429,434,4.0,828124615
66717,429,590,5.0,828124615
66716,429,588,5.0,828124615
...,...,...,...,...
81475,514,187031,2.5,1537674927
81477,514,187595,3.0,1537674946
81336,514,5247,2.5,1537757040
81335,514,5246,1.5,1537757059


No se permiten valores nulos ni na se eliminan. 

In [19]:
ratings.isnull().any()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [20]:
ratings.isna().any()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [21]:
ratings.dropna()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


## Transformación

In [22]:
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit="s")
ratings.to_parquet('../data/02_processed/ratings_clean.parquet', index=False)